In [1]:
import soundfile as sf
import os
import sys
import ffmpeg
import kagglehub
import pandas as pd
import torch
import torchaudio
from datasets import load_dataset
import numpy as np
%matplotlib inline

device = "cuda" if torch.cuda.is_available() else "cpu"

In [7]:
from huggingface_hub import login

# Replace with your actual token
login()

# Load Sample Speakers

From the Mozilla Common Voice dataset, find different speakers with Indian, French, and German accents, split by gender, speaking rate, and pitch.

In [3]:
# Utility Function
def convert_mp3_to_wav(mp3_file_path, wav_output_path):
    stream = ffmpeg.input(mp3_file_path)
    stream = ffmpeg.output(stream, wav_output_path, acodec='pcm_s16le', ac=1, ar='22050')
    ffmpeg.run(stream, overwrite_output=True)

In [4]:
%%time

from datasets import load_dataset
from collections import defaultdict

# Stream the dataset
dataset = load_dataset("mozilla-foundation/common_voice_17_0", "en", split="validation", streaming=True, trust_remote_code=True)

total = 0
empty_accents = 0
accent_counts = defaultdict(int)

for sample in dataset:
    total += 1
    accent = sample.get("accent")
    if accent:
        accent_counts[accent] += 1
    else:
        empty_accents += 1

non_empty_accents = sum(accent_counts.values())

print(f"non_empty: {non_empty_accents}, empty: {empty_accents}, total: {total}")
print("\nAccent distribution (non-empty only):")
for accent, count in sorted(accent_counts.items(), key=lambda x: -x[1]):
    print(f"{accent:20}: {count}")


README.md:   0%|          | 0.00/12.7k [00:00<?, ?B/s]

common_voice_17_0.py:   0%|          | 0.00/8.19k [00:00<?, ?B/s]

languages.py:   0%|          | 0.00/3.92k [00:00<?, ?B/s]

release_stats.py:   0%|          | 0.00/132k [00:00<?, ?B/s]

Reading metadata...: 16393it [00:00, 36017.59it/s]


non_empty: 1975, empty: 14418, total: 16393

Accent distribution (non-empty only):
United States English: 893
India and South Asia (India, Pakistan, Sri Lanka): 357
England English     : 208
Canadian English    : 87
Southern African (South Africa, Zimbabwe, Namibia): 38
Australian English  : 33
Irish English       : 29
Hong Kong English   : 24
Filipino            : 15
New Zealand English : 14
United States English,England English: 12
Scottish English    : 10
Malaysian English   : 9
Singaporean English : 9
Welsh English       : 9
spanish native with a good level of proficiency in english,colombian: 6
West Indies and Bermuda (Bahamas, Bermuda, Jamaica, Trinidad): 6
English as second language (russian as first): 6
Finnish             : 6
French              : 6
french accent       : 6
Neutral,indian,slow : 3
Scottish English,spanish english: 3
United States English,England English,India and South Asia (India, Pakistan, Sri Lanka): 3
South Indian,India and South Asia (India, Pakistan, Sri 

In [5]:
%%time

import os
import json
from datasets import load_dataset
from collections import defaultdict

# Local folder for samples and checkpoints
save_dir = "./accent_samples"
os.makedirs(save_dir, exist_ok=True)
checkpoint_path = os.path.join(save_dir, "accent_checkpoint.json")

# Accents to collect
selected_accents = ['India and South Asia (India, Pakistan, Sri Lanka)', 
                    'Australian English', 
                    'Hong Kong English']
target_per_accent = 24

# Load previous checkpoint if available
if os.path.exists(checkpoint_path):
    with open(checkpoint_path, "r") as f:
        accented_samples = defaultdict(list, json.load(f))
    print("Checkpoint loaded.")
else:
    accented_samples = defaultdict(list)

# Track current sample counts
current_counts = {a: len(accented_samples[a]) for a in selected_accents}
print("Current sample counts:", current_counts)

# Stream the dataset
dataset = load_dataset("mozilla-foundation/common_voice_17_0", "en", split="validation", streaming=True, trust_remote_code=True)

sample_counter = 0
checkpoint_interval = 10000  # how often to save progress

for sample in dataset:
    sample_counter += 1

    # Save checkpoint occasionally
    if sample_counter % checkpoint_interval == 0:
        print("Checkpointing..")
        with open(checkpoint_path, "w") as f:
            json.dump(accented_samples, f)
        print(f"Checkpoint saved at {sample_counter} samples...")

    accent = sample.get("accent")
    if accent not in selected_accents:
        continue
    if sample.get("up_votes", 0) < 2 or sample.get("down_votes", 0) > 0:
        continue
    if len(accented_samples[accent]) >= target_per_accent:
        continue

    # Store only relevant fields
    minimal_sample = {
        "path": sample.get("path"),
        "sentence": sample.get("sentence"),
        "accent": accent,
        "client_id": sample.get("client_id"),
        "waveform": sample["audio"]["array"].tolist(),
        "sampling_rate": sample["audio"]["sampling_rate"]
    }
    accented_samples[accent].append(minimal_sample)
    # Done?
    if all(len(accented_samples[a]) >= target_per_accent for a in selected_accents):
        print("All target accents collected!")
        break

# Final checkpoint
with open(checkpoint_path, "w") as f:
    json.dump(accented_samples, f)
print("Final checkpoint saved. Processed {sample_counter} samples.")

# Summary
for accent in selected_accents:
    print(f"{accent}: {len(accented_samples[accent])} samples")


Current sample counts: {'India and South Asia (India, Pakistan, Sri Lanka)': 0, 'Australian English': 0, 'Hong Kong English': 0}


Reading metadata...: 16393it [00:00, 66282.57it/s]


Checkpointing..
Checkpoint saved at 10000 samples...
Final checkpoint saved. Processed {sample_counter} samples.
India and South Asia (India, Pakistan, Sri Lanka): 24 samples
Australian English: 24 samples
Hong Kong English: 18 samples
CPU times: user 1min 4s, sys: 3.81 s, total: 1min 8s
Wall time: 1min 32s


In [6]:
# sanity check

from IPython.display import Audio, display
import random

# Load previous checkpoint
if os.path.exists(checkpoint_path):
    with open(checkpoint_path, "r") as f:
        accented_samples = defaultdict(list, json.load(f))
    print("Checkpoint loaded.")

test_sample = random.choice(accented_samples['India and South Asia (India, Pakistan, Sri Lanka)'])
display(Audio(np.array(test_sample["waveform"]), rate=test_sample["sampling_rate"]))

test_sample = random.choice(accented_samples['Australian English'])
display(Audio(np.array(test_sample["waveform"]), rate=test_sample["sampling_rate"]))

test_sample = random.choice(accented_samples['Hong Kong English'])
display(Audio(np.array(test_sample["waveform"]), rate=test_sample["sampling_rate"]))


Checkpoint loaded.
